# pLM Covariance Pooling — Remaining Experiments (LA + matrix-power)

**PP1 SoSe2026 | TU München**  
Team: Joel Simon, Julius Schmidt, Lisa Börner, Andreas Weitz

The core grid (mean / cov_supervised / cov_unsupervised / cov_pca / hybrid) is already done and plotted in the README. This notebook fills in the **four new methods** on both tasks:

| Config suffix | Method | What it adds |
|---|---|---|
| `light_attention`      | Light Attention (Stärk et al. 2021) | first-order attention pool, the literature baseline |
| `cov_supervised_sqrt`  | supervised cov + `C → C^{1/2}`       | matrix-power (iSQRT-COV) normalisation |
| `attention_cov`        | LA-weighted covariance               | per-residue attention weights in the cov |
| `attention_cov_sqrt`   | LA + cov + matrix-power              | the full stack, all three extensions |

Unlike the core grid, **none of these need a pre-fitted PCA / autoencoder checkpoint** — they are all trained end-to-end (or are the sqrt variant of the supervised cov). So this notebook is just: set up → run → mirror results.

### Workflow
1. GPU check
2. Mount Drive
3. Configuration (Drive paths)
4. Install repo
5. Wire Drive into `data/embeddings/` and `data/raw/`
6. Run the four new methods × two tasks
7. Mirror results back to Drive
8. Quick look at the summaries

## 1 · GPU check

In [ ]:
import torch

if not torch.cuda.is_available():
    print('No GPU detected — probe training will be slow on CPU.\n'
          'Runtime -> Change runtime type -> GPU recommended.')
else:
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU:  {gpu.name}')
    print(f'VRAM: {gpu.total_memory / 1e9:.1f} GB')
    print(f'CUDA: {torch.version.cuda}')

## 2 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print('Top-level folders in MyDrive:')
for f in sorted(os.listdir('/content/drive/MyDrive')):
    print(f'  {f}')

## 3 · Configuration

Same two artefacts the core-grid notebook used — if you ran that one, these paths already exist on your Drive:

1. **Embeddings** — shortcut to the shared `embeddings/` folder at `MyDrive/embeddings/`.
2. **Labels** — `raw-20260516T183459Z-3-001.zip` at `MyDrive/...`.

Adjust the paths below if your Drive names differ.

In [ ]:
from pathlib import Path

# ───────────────────────────────────────────────────────────
EMB_DIR     = Path('/content/drive/MyDrive/embeddings')                       # shared shortcut
LABELS_ZIP  = Path('/content/drive/MyDrive/raw-20260516T183459Z-3-001.zip')   # uploaded once
CODE_ZIP    = Path('/content/drive/MyDrive/sop_repo.zip')                     # fallback if git clone fails
RESULTS_OUT = Path('/content/drive/MyDrive/pp1_sop_results')                  # where runs/ lands
# ───────────────────────────────────────────────────────────

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

checks = {
    'EMB_DIR (folder)':  (EMB_DIR, EMB_DIR.is_dir()),
    'LABELS_ZIP (file)': (LABELS_ZIP, LABELS_ZIP.is_file()),
}
print(f'{"Item":<22} {"Status":<10} Path')
print('-' * 90)
for name, (path, ok) in checks.items():
    print(f'{name:<22} {"OK" if ok else "MISSING":<10} {path}')

if EMB_DIR.is_dir():
    found = sorted(EMB_DIR.iterdir())
    print(f'\nEMB_DIR contains {len(found)} files: {[p.name for p in found]}')

## 4 · Install repo

Tries `git clone` first (works if the repo is public). Falls back to `sop_repo.zip` from Drive or a local upload picker.

To build the zip locally: `zip -r sop_repo.zip src scripts configs pyproject.toml README.md tests`

In [ ]:
import subprocess, sys, shutil, zipfile
from pathlib import Path

REPO_URL = 'https://github.com/Julius-Schmidt/pLM-covariance-pooling.git'
REPO_DIR = Path('/content/pLM-covariance-pooling')
LOCAL_ZIP = Path('/content/sop_repo.zip')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# Try public git clone first — fast and always up to date.
result = subprocess.run(
    ['git', 'clone', '--depth=1', REPO_URL, str(REPO_DIR)],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print(f'Cloned from {REPO_URL}')
else:
    # Fall back to zip (Drive → local → picker).
    if CODE_ZIP.is_file():
        src_zip = CODE_ZIP
        print(f'Using zip from Drive: {CODE_ZIP}')
    elif LOCAL_ZIP.is_file():
        src_zip = LOCAL_ZIP
        print(f'Using local zip: {LOCAL_ZIP}')
    else:
        from google.colab import files
        print('Select sop_repo.zip ...')
        uploaded = files.upload()
        src_zip = Path(f'/content/{next(iter(uploaded))}')

    REPO_DIR.mkdir(parents=True)
    with zipfile.ZipFile(src_zip) as zf:
        zf.extractall(REPO_DIR)

    # Flatten if the zip added a top-level folder.
    if not (REPO_DIR / 'pyproject.toml').exists():
        inner = next(
            (p for p in REPO_DIR.iterdir() if p.is_dir() and (p / 'pyproject.toml').exists()),
            None,
        )
        if inner is None:
            raise RuntimeError('pyproject.toml not found in zip — check archive layout.')
        for item in inner.iterdir():
            shutil.move(str(item), REPO_DIR / item.name)
        inner.rmdir()
    print('Extracted from zip.')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
    cwd=str(REPO_DIR), check=True,
)
print(f'Package installed at {REPO_DIR}')

## 5 · Wire Drive into the repo

Configs use relative paths (`data/embeddings/*.h5`, `data/raw/*/`). This cell symlinks `data/embeddings/` → the shared Drive shortcut and extracts the labels zip into `data/raw/`. (No `models/` link needed — these methods load no checkpoints.)

In [ ]:
import shutil, zipfile
from pathlib import Path

DATA_DIR   = REPO_DIR / 'data'
EMB_LINK   = DATA_DIR / 'embeddings'
RAW_LINK   = DATA_DIR / 'raw'
LABELS_OUT = Path('/content/labels')

DATA_DIR.mkdir(exist_ok=True)

def relink(link: Path, target: Path):
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(target)

# embeddings → Drive shortcut
relink(EMB_LINK, EMB_DIR)

# raw → extract labels zip locally, find the deeploc/meltome parent, symlink to it
if LABELS_OUT.exists():
    shutil.rmtree(LABELS_OUT)
LABELS_OUT.mkdir()
with zipfile.ZipFile(LABELS_ZIP) as zf:
    zf.extractall(LABELS_OUT)
raw_root = next(p.parent for p in LABELS_OUT.rglob('deeploc') if p.is_dir())
relink(RAW_LINK, raw_root)

# Verify the eight key paths the configs reference
for rel in ['data/embeddings/deeploc_train.h5',
            'data/embeddings/deeploc_test.h5',
            'data/embeddings/meltome_train.h5',
            'data/embeddings/meltome_test.h5',
            'data/raw/deeploc/train_labels.csv',
            'data/raw/deeploc/test_labels.csv',
            'data/raw/meltome/train_labels.csv',
            'data/raw/meltome/test_labels.csv']:
    full = REPO_DIR / rel
    print(f'{rel}: {"OK" if full.exists() else "MISSING"}')

## 6 · Run the remaining experiments

Four new methods × two tasks = 8 configs, each over 3 seeds. Each writes one JSON to `results/runs/`.

- Leave `DC_SWEEP = None` to run every config at its default `dc = 32` (matches the headline numbers).
- Set `DC_SWEEP = [8, 16, 24, 32, 48]` to sweep the bottleneck on the **covariance** methods (`cov_supervised_sqrt`, `attention_cov`, `attention_cov_sqrt`). `light_attention` has no `dc`, so the sweep is skipped for it automatically.

In [ ]:
import subprocess, sys

CONFIGS = [
    'configs/scl/light_attention.yaml',        # LA (first-order baseline)
    'configs/scl/cov_supervised_sqrt.yaml',    # cov + matrix-power (C → C^{1/2})
    'configs/scl/attention_cov.yaml',          # LA-weighted covariance
    'configs/scl/attention_cov_sqrt.yaml',     # LA + cov + matrix-power (full stack)
    'configs/meltome/light_attention.yaml',
    'configs/meltome/cov_supervised_sqrt.yaml',
    'configs/meltome/attention_cov.yaml',
    'configs/meltome/attention_cov_sqrt.yaml',
]

DC_SWEEP = None   # or [8, 16, 24, 32, 48] to sweep the covariance methods

def stream(cmd, cwd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, cwd=str(cwd))
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed ({proc.returncode}): {" ".join(cmd)}')

for cfg in CONFIGS:
    print(f'\n=== {cfg} ===')
    cmd = [sys.executable, str(REPO_DIR / 'scripts' / 'run_experiment.py'),
           '--config', cfg]
    # light_attention has no bottleneck — don't pass --dc to it.
    if DC_SWEEP and 'light_attention' not in cfg:
        cmd += ['--dc'] + [str(d) for d in DC_SWEEP]
    stream(cmd, cwd=REPO_DIR)

## 7 · Mirror results to Drive

Copies every JSON in `results/` to Drive so it survives the session. Hand these back to whoever owns the repo — dropping them next to the existing core-grid JSONs and re-running `scripts/make_plots.py` regenerates the figures + `summary_table.tsv` with the new rows included.

In [ ]:
import shutil

src = REPO_DIR / 'results'
if not src.exists():
    print(f'No results dir at {src} — nothing to copy.')
else:
    RESULTS_OUT.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in src.rglob('*'):
        if p.is_file():
            rel = p.relative_to(src)
            dst = RESULTS_OUT / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
            n += 1
    print(f'Copied {n} files to {RESULTS_OUT}')

## 8 · Quick look at the summaries

In [ ]:
import json

runs = sorted((REPO_DIR / 'results' / 'runs').glob('*.json'))
for r in runs:
    if r.stem.endswith('_sweep'):
        continue
    d = json.loads(r.read_text())
    metric = 'accuracy_mean' if d['task'] == 'classification' else 'spearman_r_mean'
    std    = metric.replace('_mean', '_std')
    pn = ' +sqrt' if d.get('power_norm') else ''
    print(f'{d["config"]:<28} {d["method"]:<16}{pn:<6} dc={str(d.get("dc")):<5} '
          f'{metric.replace("_mean",""):<10}={d[metric]:.4f} ± {d[std]:.4f}')